# Merge HAP, energy, and biomass data

This notebook creates a country-year panel and standardizes all energy quantities to ktoe.

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
root = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
hap_dir = root / "data" / "IHME_GBD_2023_AIR_POLLUTION_1990_2023_HAP"
energy_file = root / "data" / "energy" / "EnergyConsumption_1970_to_2019 raw v2.csv"
biomass_dir = root / "data" / "energy" / "biomass"

## HAP panel

In [ ]:
hap_files = sorted(hap_dir.glob("*_HAP_[12][0-9][0-9][0-9]_Y*.CSV"))
hap = pd.concat((pd.read_csv(f, low_memory=False) for f in hap_files), ignore_index=True)

In [ ]:
hap_names = {87: "solid_fuels", 394: "coal_charcoal", 395: "wood", 396: "residue", 397: "dung"}
hap = hap.query("measure_id == 18 and sex_id == 3 and age_group_id == 22 and rei_id in @hap_names").copy()
hap = hap[hap.ihme_loc_id.astype(str).str.fullmatch(r"[A-Z]{3}")]
hap["hap_type"] = hap.rei_id.map(hap_names)

In [ ]:
population = hap.groupby(["ihme_loc_id", "location_name", "year_id"], as_index=False).population.first()
hap_panel = hap.pivot(
    index=["ihme_loc_id", "location_name", "year_id"], columns="hap_type", values="mean"
).add_prefix("hap_").reset_index()
hap_panel = hap_panel.merge(population, on=["ihme_loc_id", "location_name", "year_id"])
hap_panel = hap_panel.rename(columns={"ihme_loc_id": "iso3", "year_id": "year"})

## Existing energy data

In [ ]:
energy = pd.read_csv(energy_file, low_memory=False)
energy = energy.rename(columns={"country_code": "iso3", "FuelType": "fuel", "res": "fc"})
energy_panel = energy.pivot(index=["iso3", "year"], columns="fuel", values="fc")
energy_panel = energy_panel.add_prefix("res_").add_suffix("_ktoe").reset_index()

## Household biomass data

UN standard conversion factors are 9.768 TJ per thousand m³ of fuelwood and 28.8888 TJ per thousand metric tonnes of charcoal. One ktoe equals 41.868 TJ.

In [ ]:
biomass_specs = {
    "fuelwood": ("undata_fuelwood_households.csv", 9.768),
    "animal_waste": ("undata_animal_waste_households.csv", 1.0),
    "vegetal_residues": ("undata_vegetal_residues_households.csv", 1.0),
    "charcoal": ("undata_charcoal_households.csv", 28.8888),
}

In [ ]:
frames = []
for fuel, (filename, tj_factor) in biomass_specs.items():
    part = pd.read_csv(biomass_dir / filename)
    part["year"] = pd.to_numeric(part.Year, errors="coerce")
    part = part.dropna(subset=["year", "Quantity"]).copy()
    part["fuel"] = fuel
    part["ktoe"] = part.Quantity * tj_factor / 41.868
    frames.append(part)
biomass = pd.concat(frames, ignore_index=True)

Country names are matched to IHME codes. Former countries and unmatched territories are left unmapped rather than assigned to modern states.

In [ ]:
name_to_iso3 = hap[["location_name", "ihme_loc_id"]].drop_duplicates().set_index("location_name").ihme_loc_id.to_dict()
aliases = {
    "Bolivia (Plur. State of)": "BOL", "Central African Rep.": "CAF",
    "Dem. Rep. of the Congo": "COD", "Iran (Islamic Rep. of)": "IRN",
    "Korea, Dem.Ppl's.Rep.": "PRK", "Korea, Republic of": "KOR",
    "Lao People's Dem. Rep.": "LAO", "Micronesia (Fed. States of)": "FSM",
    "Netherlands (Kingd. of the)": "NLD", "St. Kitts-Nevis": "KNA",
    "St. Lucia": "LCA", "St. Vincent-Grenadines": "VCT",
    "State of Palestine": "PSE", "United Rep. of Tanzania": "TZA",
    "United States": "USA", "United States Virgin Is.": "VIR",
    "Venezuela (Bolivar. Rep.)": "VEN"
}

In [ ]:
biomass["iso3"] = biomass["Country or Area"].map(name_to_iso3).fillna(biomass["Country or Area"].map(aliases))
match_summary = biomass.groupby("fuel").iso3.agg(rows="size", matched="count")
match_summary["match_rate"] = match_summary.matched / match_summary.rows
match_summary

,rows,matched,match_rate
fuel,,,
animal_waste,284,283,0.996479
charcoal,4713,4283,0.908763
fuelwood,6346,5799,0.913804
vegetal_residues,1876,1876,1.000000


In [ ]:
biomass_panel = biomass.dropna(subset=["iso3"]).pivot(
    index=["iso3", "year"], columns="fuel", values="ktoe"
).add_prefix("hh_").add_suffix("_ktoe").reset_index()
biomass_panel["year"] = biomass_panel.year.astype(int)

In [ ]:
bio_cols = ["hh_fuelwood_ktoe", "hh_animal_waste_ktoe", "hh_vegetal_residues_ktoe"]
biomass_panel["biomass_components_reported"] = biomass_panel[bio_cols].notna().sum(axis=1)
biomass_panel["hh_biomass_observed_ktoe"] = biomass_panel[bio_cols].sum(axis=1, min_count=1)

The biomass total sums only reported components. Use `biomass_components_reported` to avoid treating missing fuels as zero.

In [ ]:
panel = hap_panel.merge(energy_panel, on=["iso3", "year"], how="left")
panel = panel.merge(biomass_panel, on=["iso3", "year"], how="left")
panel.shape

(6936, 34)

In [ ]:
output_dir = root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
panel.to_csv(output_dir / "hap_energy_biomass_panel.csv", index=False)

In [ ]:
pd.Series({
    "countries": panel.iso3.nunique(), "years": panel.year.nunique(),
    "rows": len(panel), "rows with residential coal": panel.res_coa_ktoe.notna().sum(),
    "rows with residential biomass": panel.res_bio_ktoe.notna().sum(),
    "rows with fuelwood": panel.hh_fuelwood_ktoe.notna().sum(),
    "rows with vegetal residues": panel.hh_vegetal_residues_ktoe.notna().sum(),
    "rows with animal waste": panel.hh_animal_waste_ktoe.notna().sum()
})

countries                         204
years                              34
rows                             6936
rows with residential coal       4105
rows with residential biomass    4105
rows with fuelwood               5734
rows with vegetal residues       1863
rows with animal waste            282
dtype: int64

In [ ]:
panel.query("year == 2019").head()

,iso3,location_name,year,hap_coal_charcoal,hap_dung,hap_residue,hap_solid_fuels,hap_wood,population,res_bdi_ktoe,...,res_oop_ktoe,res_ore_ktoe,res_sol_ktoe,res_wnd_ktoe,hh_animal_waste_ktoe,hh_charcoal_ktoe,hh_fuelwood_ktoe,hh_vegetal_residues_ktoe,biomass_components_reported,hh_biomass_observed_ktoe
29,AFG,Afghanistan,2019,0.005760,0.158454,0.224001,0.637604,0.168572,3.343500e+07,NaN,...,NaN,NaN,NaN,NaN,NaN,99.787385,246.571075,NaN,1.0,246.571075
63,AGO,Angola,2019,0.129111,0.000029,0.004124,0.375735,0.257790,3.178480e+07,0.0,...,0.0,0.0,0.0,0.0,NaN,800.258676,1942.061826,2356.782244,2.0,4298.844070
97,ALB,Albania,2019,0.001109,0.000009,0.001077,0.214275,0.263894,2.610937e+06,0.0,...,0.0,0.0,7.8,0.0,NaN,0.275999,127.246001,NaN,1.0,127.246001
131,AND,Andorra,2019,0.001264,0.000004,0.000001,0.004822,0.005356,8.169671e+04,NaN,...,NaN,NaN,NaN,NaN,NaN,0.169049,0.795336,NaN,1.0,0.795336
165,ARE,United Arab Emirates,2019,0.000010,0.000007,0.000014,0.000219,0.000004,9.840822e+06,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
